## CH3-R lights - Interactive Control Panel & Remote Mimic

### Import required libraries

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
import serial
import serial.tools.list_ports
import threading
import time
import json
import re

### Function to send serial command

In [3]:
def send_serial_command(command):
    """
    Send a command to a device through a USB to UART module
    
    Parameters:
    - command: The command to send (string or bytes)
    
    Returns:
    - Response from the device (if any)
    """
    try:       
        # Convert string to bytes if needed
        if isinstance(command, str):
            command = command.encode()
        
        # Send the command
        ser.write(command)
        
        # Wait for response (if expected)
        time.sleep(0.05)  # Short delay to ensure device processes command
        
        # Read response if available
        response = b''
        while ser.in_waiting > 0:
            response += ser.read(ser.in_waiting)
        
        return response.decode(errors='ignore')
    
    except serial.SerialException as e:
        print(f"Error: {e}")
        return None

In [4]:
serial_buffer = ""
output = widgets.Output()

def uart_listener():
    global serial_buffer
    while ser.is_open:
        try:
            if ser.in_waiting:
                data = ser.read(ser.in_waiting).decode(errors='ignore')
                if data:
                    # Print raw data in output cell
                    with output:
                        print(data, end="")
                    
                    serial_buffer += data
                    while "\n" in serial_buffer:
                        line, serial_buffer = serial_buffer.split("\n", 1)
                        line = line.strip()
                        match = re.search(r'(\{.*\})', line)
                        if match:
                            try:
                                state = json.loads(match.group(1))
                                update_ui_from_state(state)
                            except Exception as e:
                                with output:
                                    print(f"Error parsing status JSON: {e}")
        except Exception:
            pass
        time.sleep(0.05)

def query_poller():
    while ser.is_open:
        try:
            ser.write(b"#?!")
        except Exception:
            pass
        time.sleep(0.25) # Poll status every 250ms

### Functions for Legs/Ring

In [5]:
def send_leg_action(param1_radio, param2_radio, color_input, brightness_input, speed_input, attribute_input):
    # Get indexes of selected options
    index1 = list(param1_radio.options).index(param1_radio.value)
    index2 = list(param2_radio.options).index(param2_radio.value)
    
    # Correct prefix syntax to match C++ parseCommand: #L{index}_{action}
    result_string = f"#L{index1}_{index2}"
    
    # Add optional parameters if provided
    if color_input.value:
        color = color_input.value.replace('#', '').upper()
        result_string += f"_C={color}"
    
    if brightness_input.value > 0:
        result_string += f"_B={brightness_input.value}"
    
    if speed_input.value > 0:
        result_string += f"_S={speed_input.value}"
        
    if attribute_input.value > 0:
        result_string += f"_A={attribute_input.value}"
    
    result_string += "!"
    return result_string

In [6]:
def send_ring_action(param1_radio, param2_radio, color_input, brightness_input, speed_input, attribute_input):
    index1 = list(param1_radio.options).index(param1_radio.value)
    index2 = list(param2_radio.options).index(param2_radio.value)
    
    # Correct prefix syntax to match C++ parseCommand: #R{index}_{action}
    result_string = f"#R{index1}_{index2}"
    
    # Add optional parameters if provided
    if color_input.value:
        color = color_input.value.replace('#', '').upper()
        result_string += f"_C={color}"
    
    if brightness_input.value > 0:
        result_string += f"_B={brightness_input.value}"
    
    if speed_input.value > 0:
        result_string += f"_S={speed_input.value}"
        
    if attribute_input.value > 0:
        result_string += f"_A={attribute_input.value}"
    
    result_string += "!"
    return result_string

### Functions for buttons

In [7]:
is_updating_ui = False
last_interaction_time = 0

def send_command(command):
    if is_updating_ui:
        return None # Prevent circular feedback when syncing UI from robot
    response = send_serial_command(command)
    return response

In [8]:
def on_leg_input_change(change):
    global last_interaction_time
    last_interaction_time = time.time()
    if is_updating_ui:
        return
    with output:
        clear_output(wait=True)
        result = send_leg_action(leg_option1, leg_option2, leg_color, leg_brightness, leg_speed, leg_attribute)
        print(f"[SENT] Leg Action: {result}")
        response = send_command(result)

def on_ring_input_change(change):
    global last_interaction_time
    last_interaction_time = time.time()
    if is_updating_ui:
        return
    with output:
        clear_output(wait=True)
        result = send_ring_action(ring_option1, ring_option2, ring_color, ring_brightness, ring_speed, ring_attribute)
        print(f"[SENT] Ring Action: {result}")
        response = send_command(result)

### GUI Layout (widgets)

In [9]:
def make_radio_autoheight(options, description):
    height_per_option = 28
    padding = 40
    total_height = height_per_option * len(options) + padding

    return widgets.RadioButtons(
        options=options,
        description=description,
        layout=widgets.Layout(
            width='280px',
            height=f'{total_height}px',
            overflow_y='visible'
        )
    )

### Define radio buttons and parameter inputs

In [10]:
leg_option1 = make_radio_autoheight(
    options=['0 (All)', '1', '2', '3', '4', '5', '6'],
    description='Select Leg:'
)

leg_option2 = make_radio_autoheight(
    options=[
        '0 - LEG_OFF', '1 - LEG_COLOR', '2 - LEG_COLOR_FADING',
        '3 - LEG_COLOR_TOUCH', '4 - LEG_COLOR_TOUCH_FADING', 
        '5 - LEG_COLOR_UNTOUCH', '6 - LEG_COLOR_UNTOUCH_FADING', '7 - LEG_COLOR_FROM_RING'
    ],
    description='Leg Function:'
)

ring_option1 = make_radio_autoheight(
    options=['0 (n.a.)', '1'],
    description='Select Ring:'
)

ring_option2 = make_radio_autoheight(
    options=[
        '0 - RING_OFF', '1 - RING_COLOR', '2 - RING_COLOR_FADING',
        '3 - RING_METEORTAIL', '4 - RING_RAINBOW', '5 - RING_BREATH', '6 - RING_DIRECTION'
    ],
    description='Ring Function:'
)

# Optional parameter inputs for Legs
leg_color = widgets.Text(
    value='',
    placeholder='FF0000',
    description='Color (hex):',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

leg_brightness = widgets.IntSlider(
    value=0,
    min=0,
    max=255,
    step=1,
    description='Brightness:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

leg_speed = widgets.IntSlider(
    value=0,
    min=0,
    max=255,
    step=1,
    description='Speed:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

leg_attribute = widgets.IntSlider(
    value=5,
    min=0,
    max=10,
    step=1,
    description='Attribute:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

# Optional parameter inputs for Ring
ring_color = widgets.Text(
    value='',
    placeholder='00FF00',
    description='Color (hex):',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

ring_brightness = widgets.IntSlider(
    value=0,
    min=0,
    max=255,
    step=1,
    description='Brightness:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

ring_speed = widgets.IntSlider(
    value=0,
    min=0,
    max=255,
    step=1,
    description='Speed:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

ring_attribute = widgets.IntSlider(
    value=20,
    min=1,
    max=60,
    step=1,
    description='Attribute:',
    continuous_update=False,
    layout=widgets.Layout(width='250px')
)

# Attach observers for auto-reactive updates
leg_option1.observe(on_leg_input_change, names='value')
leg_option2.observe(on_leg_input_change, names='value')
leg_color.observe(on_leg_input_change, names='value')
leg_brightness.observe(on_leg_input_change, names='value')
leg_speed.observe(on_leg_input_change, names='value')
leg_attribute.observe(on_leg_input_change, names='value')

ring_option1.observe(on_ring_input_change, names='value')
ring_option2.observe(on_ring_input_change, names='value')
ring_color.observe(on_ring_input_change, names='value')
ring_brightness.observe(on_ring_input_change, names='value')
ring_speed.observe(on_ring_input_change, names='value')
ring_attribute.observe(on_ring_input_change, names='value')


### Virtual Leg Buttons (Foot Sensors Mimic)

Configure a switch to select between the **Physical Leg Buttons** or **Virtual Leg Buttons** driven by this notebook.

In [11]:
use_v_sensors_toggle = widgets.ToggleButton(
    value=False,
    description='Enable Virtual Sensors',
    button_style='danger',
    layout=widgets.Layout(width='250px')
)

v_leg_toggles = [
    widgets.ToggleButton(
        value=False,
        description=f'Leg {i}',
        button_style='danger',
        layout=widgets.Layout(width='90px')
    ) for i in range(1, 7)
]

v_legs_all = widgets.ToggleButton(
    value=False,
    description='All Legs',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

def on_v_sensors_mode_change(change):
    with output:
        clear_output(wait=True)
        val = 1 if change['new'] else 0
        cmd = f"#V0_{val}!"
        print(f"[SENT] Sensor Mode Command: {cmd}")
        send_command(cmd)
        
        # Toggle description and style dynamically
        if change['new']:
            change['owner'].description = 'Disable Virtual Sensors'
            change['owner'].button_style = 'success'
        else:
            change['owner'].description = 'Enable Virtual Sensors'
            change['owner'].button_style = 'danger'

use_v_sensors_toggle.observe(on_v_sensors_mode_change, names='value')

def make_leg_handler(leg_idx):
    def leg_handler(change):
        # Red when unpressed (danger), Green when pressed (success)
        change['owner'].button_style = 'success' if change['new'] else 'danger'
        if not use_v_sensors_toggle.value:
            return # Block sending serial commands when virtual sensors is off
        with output:
            clear_output(wait=True)
            val = 1 if change['new'] else 0
            cmd = f"#V{leg_idx}_{val}!"
            print(f"[SENT] Virtual Leg {leg_idx} Sensor: {cmd}")
            send_command(cmd)
    return leg_handler

for idx, btn in enumerate(v_leg_toggles, start=1):
    btn.observe(make_leg_handler(idx), names='value')

def on_all_legs_change(change):
    change['owner'].button_style = 'success' if change['new'] else 'danger'
    if not use_v_sensors_toggle.value:
        return
    # Proactively change all individual toggle buttons
    for btn in v_leg_toggles:
        btn.value = change['new']

v_legs_all.observe(on_all_legs_change, names='value')

v_legs_hbox = widgets.HBox(v_leg_toggles + [v_legs_all], layout=widgets.Layout(justify_content='center'))
v_sensors_layout = widgets.VBox([
    widgets.HBox([use_v_sensors_toggle], layout=widgets.Layout(justify_content='center', margin='10px')),
    v_legs_hbox
], layout=widgets.Layout(border='1px dashed orange', padding='10px', margin='15px 0', border_radius='5px', width='100%'))

### Visual PS2 Remote Control Mimic (LED Mode Only)

This UI maps perfectly to the **Advanced LED Mode** buttons of the real PS2 remote control. The notebook maintains the configuration parameters and smoothly sends the updated `#R` or `#L` commands when you trigger the virtual sticks/D-Pad.

In [12]:
led_state = {
    "ring_pattern": 1,
    "ring_color_idx": 0,
    "leg_pattern": 1,
    "leg_color_idx": 0,
    "speed": 10,
    "attribute": 85
}

RING_PATTERNS = ['RING_OFF', 'RING_COLOR', 'RING_COLOR_FADING', 'RING_METEORTAIL', 'RING_RAINBOW', 'RING_BREATH']
LEG_PATTERNS = ['LEG_OFF', 'LEG_COLOR', 'LEG_COLOR_FADING', 'LEG_COLOR_TOUCH', 'LEG_COLOR_TOUCH_FADING', 'LEG_COLOR_UNTOUCH', 'LEG_COLOR_UNTOUCH_FADING', 'LEG_COLOR_FROM_RING']
COLORS = ['FFFFFF', 'FF0000', '00FF00', '0000FF', 'FFFF00', 'FF00FF', '00FFFF', 'FF8000', '8000FF', 'FF0080']

def send_sync_ring_command():
    cmd = f"#R0_{led_state['ring_pattern']}_C={COLORS[led_state['ring_color_idx']]}_S={led_state['speed']}_A={led_state['attribute']}!"
    with output:
        clear_output(wait=True)
        print(f"[SENT - Virtual L-Stick] {cmd}")
        send_command(cmd)
        # Sync values visually into sliders
        ring_option2.value = f"{led_state['ring_pattern']} - {RING_PATTERNS[led_state['ring_pattern']]}"
        ring_color.value = COLORS[led_state['ring_color_idx']]
        ring_speed.value = led_state['speed']
        ring_attribute.value = led_state['attribute']

def send_sync_leg_command():
    cmd = f"#L0_{led_state['leg_pattern']}_C={COLORS[led_state['leg_color_idx']]}_S={led_state['speed']}_A={led_state['attribute']}!"
    with output:
        clear_output(wait=True)
        print(f"[SENT - Virtual R-Stick] {cmd}")
        send_command(cmd)
        # Sync values visually into sliders
        leg_option2.value = f"{led_state['leg_pattern']} - {LEG_PATTERNS[led_state['leg_pattern']]}"
        leg_color.value = COLORS[led_state['leg_color_idx']]
        leg_speed.value = led_state['speed']
        leg_attribute.value = led_state['attribute']

# Style
btn_style = widgets.Layout(width='45px', height='40px', margin='2px')

# Left Joystick: Ring Controls
lj_up = widgets.Button(description="▲", layout=btn_style, button_style='info', tooltip='Next Color')
lj_left = widgets.Button(description="◀", layout=btn_style, button_style='info', tooltip='Prev Pattern')
lj_right = widgets.Button(description="▶", layout=btn_style, button_style='info', tooltip='Next Pattern')
lj_down = widgets.Button(description="▼", layout=btn_style, button_style='info', tooltip='Prev Color')
lj_center = widgets.Button(description="L3", layout=btn_style, button_style='primary')

def on_lj_up(b):
    led_state["ring_color_idx"] = (led_state["ring_color_idx"] + 1) % len(COLORS)
    send_sync_ring_command()
def on_lj_down(b):
    led_state["ring_color_idx"] = (led_state["ring_color_idx"] - 1 + len(COLORS)) % len(COLORS)
    send_sync_ring_command()
def on_lj_left(b):
    led_state["ring_pattern"] = (led_state["ring_pattern"] - 1 + len(RING_PATTERNS)) % len(RING_PATTERNS)
    send_sync_ring_command()
def on_lj_right(b):
    led_state["ring_pattern"] = (led_state["ring_pattern"] + 1) % len(RING_PATTERNS)
    send_sync_ring_command()

lj_up.on_click(on_lj_up)
lj_down.on_click(on_lj_down)
lj_left.on_click(on_lj_left)
lj_right.on_click(on_lj_right)

lj_box = widgets.VBox([
    widgets.HBox([widgets.Box(layout=btn_style), lj_up, widgets.Box(layout=btn_style)]),
    widgets.HBox([lj_left, lj_center, lj_right]),
    widgets.HBox([widgets.Box(layout=btn_style), lj_down, widgets.Box(layout=btn_style)])
], layout=widgets.Layout(align_items='center'))

# Right Joystick: Leg Controls
rj_up = widgets.Button(description="▲", layout=btn_style, button_style='success', tooltip='Next Color')
rj_left = widgets.Button(description="◀", layout=btn_style, button_style='success', tooltip='Prev Pattern')
rj_right = widgets.Button(description="▶", layout=btn_style, button_style='success', tooltip='Next Pattern')
rj_down = widgets.Button(description="▼", layout=btn_style, button_style='success', tooltip='Prev Color')
rj_center = widgets.Button(description="R3", layout=btn_style, button_style='primary')

def on_rj_up(b):
    led_state["leg_color_idx"] = (led_state["leg_color_idx"] + 1) % len(COLORS)
    send_sync_leg_command()
def on_rj_down(b):
    led_state["leg_color_idx"] = (led_state["leg_color_idx"] - 1 + len(COLORS)) % len(COLORS)
    send_sync_leg_command()
def on_rj_left(b):
    led_state["leg_pattern"] = (led_state["leg_pattern"] - 1 + len(LEG_PATTERNS)) % len(LEG_PATTERNS)
    send_sync_leg_command()
def on_rj_right(b):
    led_state["leg_pattern"] = (led_state["leg_pattern"] + 1) % len(LEG_PATTERNS)
    send_sync_leg_command()

rj_up.on_click(on_rj_up)
rj_down.on_click(on_rj_down)
rj_left.on_click(on_rj_left)
rj_right.on_click(on_rj_right)

rj_box = widgets.VBox([
    widgets.HBox([widgets.Box(layout=btn_style), rj_up, widgets.Box(layout=btn_style)]),
    widgets.HBox([rj_left, rj_center, rj_right]),
    widgets.HBox([widgets.Box(layout=btn_style), rj_down, widgets.Box(layout=btn_style)])
], layout=widgets.Layout(align_items='center'))

# D-Pad: Speed & Custom Attributes
dpad_up = widgets.Button(description="▲", layout=btn_style, button_style='warning', tooltip='Speed Up')
dpad_left = widgets.Button(description="◀", layout=btn_style, button_style='warning', tooltip='Attr Down')
dpad_right = widgets.Button(description="▶", layout=btn_style, button_style='warning', tooltip='Attr Up')
dpad_down = widgets.Button(description="▼", layout=btn_style, button_style='warning', tooltip='Speed Down')

def on_dpad_up(b):
    led_state["speed"] = min(led_state["speed"] + 10, 255)
    send_sync_ring_command()
    send_sync_leg_command()
def on_dpad_down(b):
    led_state["speed"] = max(led_state["speed"] - 10, 0)
    send_sync_ring_command()
    send_sync_leg_command()
def on_dpad_left(b):
    led_state["attribute"] = max(led_state["attribute"] - 10, 0)
    send_sync_ring_command()
    send_sync_leg_command()
def on_dpad_right(b):
    led_state["attribute"] = min(led_state["attribute"] + 10, 255)
    send_sync_ring_command()
    send_sync_leg_command()

dpad_up.on_click(on_dpad_up)
dpad_down.on_click(on_dpad_down)
dpad_left.on_click(on_dpad_left)
dpad_right.on_click(on_dpad_right)

dpad_box = widgets.VBox([
    widgets.HBox([widgets.Box(layout=btn_style), dpad_up, widgets.Box(layout=btn_style)]),
    widgets.HBox([dpad_left, widgets.Box(layout=btn_style), dpad_right]),
    widgets.HBox([widgets.Box(layout=btn_style), dpad_down, widgets.Box(layout=btn_style)])
], layout=widgets.Layout(align_items='center'))

remote_ui = widgets.HBox([
    widgets.VBox([widgets.Label("Left Stick (Ring Pattern & Color)"), lj_box], layout=widgets.Layout(align_items='center')),
    widgets.VBox([widgets.Label("D-Pad (Speed & Attributes)"), dpad_box], layout=widgets.Layout(align_items='center')),
    widgets.VBox([widgets.Label("Right Stick (Leg Pattern & Color)"), rj_box], layout=widgets.Layout(align_items='center'))
], layout=widgets.Layout(justify_content='space-around', margin='20px 0', border='2px solid #ccc', padding='10px', border_radius='10px', width='100%'))

### Bi-Directional State Syncing (State Parser)

We parse status strings broadcasted from the robot when polled, and automatically sync all widget states in real-time.

In [13]:
def update_ui_from_state(state):
    global is_updating_ui
    is_updating_ui = True
    try:
        # Only sync parameters if the user has not interacted with the UI in the last 2 seconds
        if time.time() - last_interaction_time > 2.0:
            # 1. Update Ring Parameters
            if "ring" in state:
                r_pat = state["ring"]["p"]
                if r_pat < len(RING_PATTERNS):
                    ring_option2.value = f"{r_pat} - {RING_PATTERNS[r_pat]}"
                    led_state["ring_pattern"] = r_pat
                
                r_col = state["ring"]["c"].replace("0x", "")
                if len(r_col) == 6:
                    ring_color.value = r_col
                    if r_col in COLORS:
                        led_state["ring_color_idx"] = COLORS.index(r_col)
                
                ring_brightness.value = state["ring"]["t"]
                # Sync newly added speed and attribute values
                if "s" in state["ring"]:
                    ring_speed.value = state["ring"]["s"]
                    led_state["speed"] = state["ring"]["s"]
                if "a" in state["ring"]:
                    ring_attribute.value = state["ring"]["a"]
                    led_state["attribute"] = state["ring"]["a"]
                
            # 2. Update Legs Parameters
            if "legs" in state and len(state["legs"]) > 0:
                l_pat = state["legs"][0]["p"]
                if l_pat < len(LEG_PATTERNS):
                    leg_option2.value = f"{l_pat} - {LEG_PATTERNS[l_pat]}"
                    led_state["leg_pattern"] = l_pat
                
                l_col = state["legs"][0]["c"].replace("0x", "")
                if len(l_col) == 6:
                    leg_color.value = l_col
                    if l_col in COLORS:
                        led_state["leg_color_idx"] = COLORS.index(l_col)
                
                # Sync leg brightness, speed, and attribute values
                if "t" in state["legs"][0]:
                    leg_brightness.value = state["legs"][0]["t"]
                if "s" in state["legs"][0]:
                    leg_speed.value = state["legs"][0]["s"]
                if "a" in state["legs"][0]:
                    leg_attribute.value = state["legs"][0]["a"]
                        
        # 3. Update Toggle Buttons if virtual leg sensors is NOT active
        if "btns" in state and not use_v_sensors_toggle.value:
            btns_str = state["btns"]
            for i in range(min(len(btns_str), 6)):
                # LOW (active) is printed as 1 (pressed)
                v_leg_toggles[i].value = (btns_str[i] == '1')
                
    except Exception:
        pass
    finally:
        is_updating_ui = False

### GUI Layout Integration

In [14]:
# Create parameter rows (swapped: Ring on Left, Legs on Right)
color_row = widgets.HBox([ring_color, leg_color], layout=widgets.Layout(justify_content='space-around', width='100%'))
brightness_row = widgets.HBox([ring_brightness, leg_brightness], layout=widgets.Layout(justify_content='space-around', width='100%'))
speed_row = widgets.HBox([ring_speed, leg_speed], layout=widgets.Layout(justify_content='space-around', width='100%'))
attribute_row = widgets.HBox([ring_attribute, leg_attribute], layout=widgets.Layout(justify_content='space-around', width='100%'))

# Top section: all 4 radio button lists side-by-side

selection_row = widgets.HBox([
    ring_option1,
    ring_option2,
    leg_option1,
    leg_option2
], layout=widgets.Layout(justify_content='space-around', width='100%', margin='0 0 -15px 0'))

# Main UI layout
ui = widgets.VBox([
    selection_row,
    color_row,
    brightness_row,
    speed_row,
    attribute_row,
    v_sensors_layout,
    widgets.HTML("<h4 style='margin: 10px 0 5px 0; padding: 0;'>Visual PS2 Controller Mimic (LED Control Mode)</h4>"),
    remote_ui,
    output
], layout=widgets.Layout(align_items='center', width='100%', padding='5px'))

### Serial communication initialization

In [18]:
# List available ports (helpful for debugging)
print("Available ports:")
ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(f" - {p.device}: {p.description}")

# Automatically pick the first available port, fallback to 'COM4'
if len(ports) > 0:
    port = ports[0].device
    print(f"Automatically selected port: {port}")
else:
    port = 'COM4'
    print(f"No active ports found. Falling back to default: {port}")

# Open serial connection
if 'ser' not in globals() or not ser.is_open:
    ser = serial.Serial(
        port=port,
        baudrate=115200,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=serial.STOPBITS_ONE,
        timeout=1
    )

print(f"\nConnected to {port} at 115200 baud")

Available ports:
 - COM6: USB Serial Device (COM6)
Automatically selected port: COM6

Connected to COM6 at 115200 baud


### Execute the test code

In [19]:
# --- Start listener thread (only once) ---
if 'listener_thread' not in globals() or not listener_thread.is_alive():
    listener_thread = threading.Thread(target=uart_listener, daemon=True)
    listener_thread.start()

# --- Start query poller thread (only once) ---
if 'poller_thread' not in globals() or not poller_thread.is_alive():
    poller_thread = threading.Thread(target=query_poller, daemon=True)
    poller_thread.start()

# --- Display ---
display(ui)

with output:
    clear_output(wait=True)
    print("System ready!")
    print("Send commands and watch for both the raw command and parsed output.\n")

### Test Examples

Try these test cases:
1. **Basic command**: Leg 1, Action 1 (no optional params)
2. **With color**: Leg 2, Action 1, Color: FF0000 (red)
3. **Full command**: Leg 3, Action 2, Color: 00FF00, Brightness: 128, Speed: 50
4. **Ring command**: Ring 1, Action 4 (rainbow), Speed: 100
5. **Virtual Sensor**: Enable virtual sensors, click Leg 1 to press/release, observe untouch LED animation!

In [17]:
# Run this cell to close the serial connection when done
if 'ser' in globals() and ser.is_open:
    ser.close()
    print("Serial connection closed.")

Serial connection closed.
